# Q1. Embedding a query

We are converting the sentence “How does approximate nearest neighbor search work?” into a 384-number vector and checking the first number, v[0].

In [3]:
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)

v[0]

2026-06-29 20:58:24.477951192 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

# Loading the data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [36]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [6]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

content = ''

for doc in documents:
    if doc['filename'] == target_filename:
        content = doc['content']
        break

In [39]:
print(content[:500])  # Print the first 500 characters of the content

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over ev


In [8]:
page_content_vector = embed.encode(content)

In [9]:
score = page_content_vector.dot(v)

In [10]:
score

np.float64(0.36107026789538205)

#  Q3. Chunking and search by hand

In [11]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [13]:
# This is called a list comprehension
chunk_texts = [chunk["content"] for chunk in chunks]

batch_vectors = embed.encode_batch(chunk_texts)

In [14]:
chunk_texts[:2]

['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

In [15]:
# stack into matrix
import numpy as np

X = np.array(batch_vectors)

In [16]:
# Score with the query vector from Q1
scores = X.dot(v)

In [17]:
# idx is the index of the chunk with the highest similarity score.
idx = np.argmax(scores)

In [18]:
idx

np.int64(94)

In [19]:
print(chunks[idx])

{'start': 1000, 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one p

In [20]:
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

# Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

For Q4, you use the same chunks and vectors X, but instead of searching by hand with:

scores = X.dot(v)

you use minsearch.VectorSearch.

In [21]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

In [22]:
query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)

In [23]:
results = index.search(v_query, num_results=5)

In [24]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

For one small search, it does not feel like it saves many steps. <br> 
he code by hand only gives best one unless you write more code. For top 5 you need more manual code. And because this this way exists.


# Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method.<br> Which file shows up in the vector results but not in the text results?

In [25]:
# 1. Vector search
query = "How do I store vectors in PostgreSQL?"
v_query = embed.encode(query)

vector_results = index.search(v_query, num_results=5)

In [26]:
vector_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [27]:
# 2. Text / keyword search
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

text_results = text_index.search(query, num_results=5)

In [28]:
vector_files = [r["filename"] for r in vector_results]
text_files = [r["filename"] for r in text_results]

vector_files, text_files

(['02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/08-pgvector.md',
  '02-vector-search/lessons/08-pgvector.md'],
 ['02-vector-search/lessons/02-embeddings.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md',
  '03-orchestration/lessons/05-rag.md',
  '02-vector-search/lessons/01-intro.md'])

In [29]:
for file in vector_files:
    if file not in text_files:
        print(file)

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


# Q6. Hybrid search


In [30]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [31]:
query = "How do I give the model access to tools?"

In [32]:
# 1. Vector search
v_query = embed.encode(query)

vector_results = index.search(
    v_query,
    num_results=5
)

In [33]:
# 2. Text search
text_results = text_index.search(
    query,
    num_results=5
)

In [34]:
# 3. Fuse with RRF
results = rrf([vector_results, text_results])

In [35]:
# See final hybrid results
for result in results:
    print(result["filename"], result["start"])

01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/14-agentic-loop.md 0
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0
